In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Nama Kelompok:**

- **Muhammad Akhdan Athallah || NIM: 2802446560**
- **Muhammad Hylmi Razzan || NIM: 2802444901**

In [ ]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
import os
import random
import math
import hashlib
from PIL import Image
from tensorflow.keras import layers, models, regularizers

# Silence unnecessary warnings from TensorFlow
tf.get_logger().setLevel(logging.ERROR)

# Configuration
SEED = 42
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Path dataset
dataset_path = '/content/drive/MyDrive/DatasetIMGProb'

# Define Classes
classes = ['Earthquake', 'Land_Slide', 'Urban_Fire', 'Water_Disaster']

## **Explanatory Data Analysis (EDA) & Preprocessing**

In [ ]:
import seaborn as sns

class_counts = {}

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)

    count = len([
        f for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    class_counts[class_name] = count

df_counts = pd.DataFrame(
    class_counts.items(),
    columns=['Class', 'Count']
)

total_images = df_counts['Count'].sum()
print(f"Total Images in Dataset: {total_images}")

plt.figure(figsize=(10, 6))

sns.barplot(
    x='Class',
    y='Count',
    hue='Class',
    data=df_counts,
    palette='viridis',
    legend=False
)

plt.title('Number of Natural Disaster Images', fontsize=14, pad=15)
plt.xlabel('Disaster Type', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)

max_count = df_counts['Count'].max()

for index, value in enumerate(df_counts['Count']):
    plt.text(
        index,
        value + (max_count * 0.02),
        str(value),
        ha='center',
        fontsize=11,
        fontweight='bold'
    )

plt.ylim(0, max_count * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
samples_per_class = 5

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)

    files = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    sample_files = random.sample(files, min(samples_per_class, len(files)))

    plt.figure(figsize=(15, 3))
    plt.suptitle(f"Class: {class_name}", fontsize=14)

    for i, fname in enumerate(sample_files):
        img_path = os.path.join(class_dir, fname)
        img = Image.open(img_path)

        plt.subplot(1, samples_per_class, i + 1)
        plt.imshow(img)
        plt.title(fname, fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
image_info = []

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)

    files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    ratios = []

    for fname in files:
        try:
            with Image.open(os.path.join(class_dir, fname)) as img:
                ratios.append(img.width / img.height)
        except:
            pass
    avg_ratio = sum(ratios) / len(ratios) if ratios else 0

    image_info.append([class_name, avg_ratio])

df_info = pd.DataFrame(image_info, columns=['Class', 'Avg_Aspect_Ratio'])

plt.figure(figsize=(10, 6))
sns.barplot(x='Class', y='Avg_Aspect_Ratio', data=df_info, palette='coolwarm')

plt.title('Average Aspect Ratio per Disaster Type', fontsize=14, pad=15)
plt.xlabel('Disaster Type', fontsize=12)
plt.ylabel('Average Aspect Ratio', fontsize=12)

max_ratio = df_info['Avg_Aspect_Ratio'].max()

for index, value in enumerate(df_info['Avg_Aspect_Ratio']):
    plt.text(index, value + (max_ratio * 0.02), f'{value:.2f}',
             ha='center', fontsize=11, fontweight='bold')

plt.ylim(0, max_ratio * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
print("Memproses analisis kecerahan gambar...")

brightness_stats = {}

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)

    files = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if not files:
        continue

    max_bright = -1
    min_bright = 256
    brightest_img_path = None
    darkest_img_path = None
    total_brightness = 0

    for fname in files:
        img_path = os.path.join(class_dir, fname)

        try:
            img = Image.open(img_path).convert('L') #Ubah gambar jadi hitam/putih

            img_array = np.array(img)

            avg_pixel_brightness = np.mean(img_array)
            total_brightness += avg_pixel_brightness

            if avg_pixel_brightness > max_bright:
                max_bright = avg_pixel_brightness
                brightest_img_path = img_path

            if avg_pixel_brightness < min_bright:
                min_bright = avg_pixel_brightness
                darkest_img_path = img_path

        except Exception as e:
            print(f"Gagal membaca {img_path}: {e}")

    class_avg_brightness = total_brightness / len(files)

    brightness_stats[class_name] = {
        'avg_brightness': class_avg_brightness,
        'brightest_path': brightest_img_path,
        'brightest_val': max_bright,
        'darkest_path': darkest_img_path,
        'darkest_val': min_bright
    }

In [ ]:
for class_name, stats in brightness_stats.items():
    print(f"\n{'-'*40}")
    print(f"Class: {class_name}")
    print(f"Average Class Brightness: {stats['avg_brightness']:.2f} (Scale 0-255)")
    print(f"{'-'*40}")

    plt.figure(figsize=(4, 2))

    plt.subplot(1, 2, 1)

    if stats['darkest_path']:
        dark_img = Image.open(stats['darkest_path'])
        plt.imshow(dark_img)
        plt.title(f"Darkest\nValue: {stats['darkest_val']:.2f}")
    plt.axis("off")

    plt.subplot(1, 2, 2)

    if stats['brightest_path']:
        bright_img = Image.open(stats['brightest_path'])
        plt.imshow(bright_img)
        plt.title(f"Brightest\nValue: {stats['brightest_val']:.2f}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
deleted_count = 0
seen_hashes = {}

print("Starting cleanup with EARTHQUAKE (05) priority...\n")

for folder_path, _, files in os.walk(dataset_path):
    for file_name in files:
        if not file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        file_path = os.path.join(folder_path, file_name)

        if os.path.getsize(file_path) == 0:
            try:
                os.remove(file_path)
                print(f"Deleted: {file_name} | Reason: Empty file (0 bytes)")
                deleted_count += 1
            except Exception: pass
            continue

        try:
            with Image.open(file_path) as img:
                img.verify()
            with Image.open(file_path) as img:
                if img.mode != 'RGB':
                    os.remove(file_path)
                    print(f"Deleted: {file_name} | Reason: Not RGB ({img.mode})")
                    deleted_count += 1
                    continue
        except Exception:
            try:
                os.remove(file_path)
                print(f"Deleted: {file_name} | Reason: Corrupt file")
                deleted_count += 1
            except Exception: pass
            continue

        with open(file_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        if file_hash in seen_hashes:
            old_file_path = seen_hashes[file_hash]
            old_file_name = os.path.basename(old_file_path)

            if file_name.startswith('05') and not old_file_name.startswith('05'):
                try:
                    os.remove(old_file_path)
                    print(f"Deleted: {old_file_name} | Reason: Overridden by '{file_name}'")
                    seen_hashes[file_hash] = file_path
                    deleted_count += 1
                except Exception: pass
            else:
                try:
                    os.remove(file_path)
                    print(f"Deleted: {file_name} | Reason: Duplicate of '{old_file_name}'")
                    deleted_count += 1
                except Exception: pass
        else:
            seen_hashes[file_hash] = file_path

print("\n" + "=" * 60)
if deleted_count == 0:
    print("CLEAN! No images were deleted. Your dataset is ready.")
else:
    print(f"Cleanup Complete! Removed a total of {deleted_count} images.")
print("=" * 60)

In [ ]:
top_n = 30

print("Starting RGB structure scan...\n")

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)

    image_data = []
    class_rgb_sum = np.array([0.0, 0.0, 0.0])
    valid_images = 0

    for file_name in os.listdir(class_dir):
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(class_dir, file_name)

            try:
                img = Image.open(img_path).convert('RGB')
                img_array = np.array(img)
                mean_rgb = np.mean(img_array, axis=(0, 1))

                image_data.append({
                    'file_name': file_name,
                    'path': img_path,
                    'mean_rgb': mean_rgb
                })

                class_rgb_sum += mean_rgb
                valid_images += 1

            except Exception as e:
                continue

    if valid_images == 0:
        continue

    class_mean_rgb = class_rgb_sum / valid_images

    for data in image_data:
        deviation = np.linalg.norm(data['mean_rgb'] - class_mean_rgb)
        data['deviation'] = deviation

    image_data.sort(key=lambda x: x['deviation'], reverse=True)

    outliers = image_data[:top_n]

    print(f"{'='*50}")
    print(f"CLASS: {class_name} | Class Average RGB: [{class_mean_rgb[0]:.1f}, {class_mean_rgb[1]:.1f}, {class_mean_rgb[2]:.1f}]")
    print(f"Displaying the {len(outliers)} most deviating images:")

    cols = 5
    rows = math.ceil(len(outliers) / cols)

    plt.figure(figsize=(20, 4 * rows))

    for i, data in enumerate(outliers):
        img = Image.open(data['path'])

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)

        plt.title(f"{data['file_name']}\nDeviation Score: {data['deviation']:.1f}", fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
delete_list = [
    '01_01_0339.png', '03_0786.png', '04_01_0443.png', '04_01_0392.png',
    '04_01_0047.png', '04_01_0345.png', '04_01_0104.png', '04_01_0138.png',
    '04_01_0348.png', '04_01_0441.png', '04_01_0451.png', '04_01_0107.png',
    '04_01_0008.png', '04_01_0102.png', '04_01_0220.png', '04_01_0069.png',
    '04_01_0218.png', '04_01_0040.png', '04_01_0276.png'
]

found_files = []

for folder_path, _, files in os.walk(dataset_path):
    for file_name in files:
        if file_name in delete_list:
            file_path = os.path.join(folder_path, file_name)
            class_name = os.path.basename(folder_path)
            found_files.append({'name': file_name, 'path': file_path, 'class_name': class_name})

if not found_files:
    print("No files found to delete.")
else:
    cols = 5
    rows = math.ceil(len(found_files) / cols)
    plt.figure(figsize=(20, 4 * rows))

    for i, data in enumerate(found_files):
        try:
            img = Image.open(data['path'])
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.title(f"{data['name']}\n({data['class_name']})", fontsize=10)
            plt.axis('off')
        except:
            pass

    plt.tight_layout()
    plt.show()

    print("=" * 50)
    deleted_count = 0
    for data in found_files:
        try:
            os.remove(data['path'])
            print(f"✅ DELETED: {data['name']}")
            deleted_count += 1
        except Exception as e:
            print(f"❌ FAILED TO DELETE {data['name']}: {e}")

    print(f"\nDone! {deleted_count} files have been permanently deleted.")
    print("=" * 50)

## **Import Data**

In [ ]:
ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    crop_to_aspect_ratio=True
).cache()

In [ ]:
# 1 Sampel Import Data
for images, labels in ds.take(1):
    sample_image = images[0].numpy().astype("uint8")
    sample_label = labels[0].numpy()
    sample_category = classes[sample_label]

    plt.figure(figsize=(2, 2))
    plt.imshow(sample_image)
    plt.title(sample_category)
    plt.axis("off")
    plt.show()

    print(f"Image Batch Shape: {images.shape}") # Batch Size, Img Height, Img Width, Channels
    print(f"Label Batch Shape: {labels.shape}")
    print(f"One Image Shape: {sample_image.shape}")
    print("Top-Left Pixel RGB:", sample_image[0][0])

In [ ]:
# Jumlah Gambar per Kelas
class_counts = {}

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)
    count = len([
        f for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    class_counts[class_name] = count

df_counts = pd.DataFrame(class_counts.items(), columns=['Class', 'Count'])
df_counts

## **Split Data (Train, Val, Test)**

In [ ]:
dataset_size = len(ds)
train_size = int(0.70 * dataset_size)
val_size = int(0.15 * dataset_size)

# Split
raw_train_ds = ds.take(train_size)
remaining_ds = ds.skip(train_size)
raw_val_ds = remaining_ds.take(val_size)
raw_test_ds = remaining_ds.skip(val_size)

# Reshuffle per epoch
raw_train_ds = raw_train_ds.shuffle(
    buffer_size=train_size,
    seed=SEED,
    reshuffle_each_iteration=True
)

print("Total batch     :", dataset_size)
print("Train batch     :", tf.data.experimental.cardinality(raw_train_ds).numpy())
print("Validation batch:", tf.data.experimental.cardinality(raw_val_ds).numpy())
print("Test batch      :", tf.data.experimental.cardinality(raw_test_ds).numpy())

In [ ]:
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
], name="data_augmentation")

def prepare_data(dataset, augment=False):
    dataset = dataset.map(
        lambda x, y: (layers.Rescaling(1./255)(x), y), # Rescale gambar jadi rentang 0 sampai 1
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if augment:
        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
# Preprocess dilakukan
train_ds = prepare_data(raw_train_ds, augment=True)
val_ds = prepare_data(raw_val_ds, augment=False)
test_ds = prepare_data(raw_test_ds, augment=False)

In [ ]:
# Sanity Check
for images, labels in train_ds.take(1):
    batch_size = images.shape[0]

    random_index = random.randint(0, batch_size - 1)

    sample_image = images[random_index].numpy()
    sample_label = labels[random_index].numpy()
    sample_category = classes[sample_label]

    plt.figure(figsize=(2, 2))
    plt.imshow(sample_image)
    plt.title(sample_category)
    plt.axis("off")
    plt.show()

    # Mencetak informasi untuk melihat bentuk data
    print(f"Indeks Acak yang Terpilih: {random_index}")
    print(f"Image Batch Shape: {images.shape}")
    print(f"Label Batch Shape: {labels.shape}")
    print(f"One Image Shape: {sample_image.shape}")

    # Ini akan mengambil nilai warna (RGB) dari piksel di baris ke-1, kolom ke-1
    print("Top-Left Pixel RGB:", sample_image[1][1])

## **Scratch Model (No Oversampling, Class Weight)**

---



In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_counts = [36, 381, 399, 1014]

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(4),
    y=np.repeat(np.arange(4), class_counts)
)

class_weight = dict(enumerate(weights))
print(class_weight)

In [ ]:
model_base1 = keras.Sequential([

    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(4, activation='softmax')

])

model_base1.summary()

In [ ]:
model_base1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

In [ ]:
history1 = model_base1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weight,
    callbacks=[lr_scheduler, early_stop],
    verbose=1
)

In [ ]:
# Plot Train
plt.plot(history1.history['loss'], label='Train Loss')
plt.plot(history1.history['val_loss'], label='Validation Loss')
plt.title('Scratch Model No Oversampling(Class Weight) Loss')
plt.legend()

plt.show()

In [ ]:
# Plot Accuracy
plt.plot(history1.history['accuracy'], label='Train Accuracy')
plt.plot(history1.history['val_accuracy'], label='Validation Accuracy')
plt.title('Scratch Model No Oversampling(Class Weight) Accuracy')
plt.legend()

plt.show()

### **Evaluasi Scratch Model No Oversampling**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

y_true_sc1 = []
y_pred_sc1 = []

for images, labels in test_ds:
    preds_sc1 = model_base1.predict(images, verbose=0)
    pred_labels_sc1 = np.argmax(preds_sc1, axis=1)

    y_true_sc1.extend(labels.numpy())
    y_pred_sc1.extend(pred_labels_sc1)

acc_sc1 = accuracy_score(y_true_sc1, y_pred_sc1)
precision_sc1 = precision_score(y_true_sc1, y_pred_sc1, average='macro', zero_division=0)
recall_sc1 = recall_score(y_true_sc1, y_pred_sc1, average='macro', zero_division=0)
f1_sc1 = f1_score(y_true_sc1, y_pred_sc1, average='macro', zero_division=0)

print("=== Scratch Model No Oversampling(Class Weight) ===")
print("Accuracy :", acc_sc1)
print("Precision:", precision_sc1)
print("Recall   :", recall_sc1)
print("F1-Score :", f1_sc1)

print(classification_report(y_true_sc1, y_pred_sc1, target_names=classes, zero_division=0))

## **Scratch Model (Oversampling)**

---



In [ ]:
class_0 = raw_train_ds.unbatch().filter(lambda x, y: y == 0)
class_1 = raw_train_ds.unbatch().filter(lambda x, y: y == 1)
class_2 = raw_train_ds.unbatch().filter(lambda x, y: y == 2)
class_3 = raw_train_ds.unbatch().filter(lambda x, y: y == 3)

In [ ]:
class_0 = class_0.repeat()
class_1 = class_1.repeat()
class_2 = class_2.repeat()
class_3 = class_3.repeat()

In [ ]:
balanced_train_ds = tf.data.Dataset.sample_from_datasets(
    [class_0, class_1, class_2, class_3],
    weights=[0.25, 0.25, 0.25, 0.25]
)

In [ ]:
balanced_train_ds = balanced_train_ds.batch(BATCH_SIZE)
train_ds_sc2 = prepare_data(balanced_train_ds, augment=True)

In [ ]:
model_base2 = keras.Sequential([

    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(4, activation='softmax')

])

model_base2.summary()

In [ ]:
model_base2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history2 = model_base2.fit(
    train_ds_sc2,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=len(raw_train_ds),
    callbacks=[lr_scheduler, early_stop],
    verbose=1
)

In [ ]:
# Plot Train
plt.plot(history2.history['loss'], label='Train Loss')
plt.plot(history2.history['val_loss'], label='Validation Loss')
plt.title('Scratch Model Oversampling Loss')
plt.legend()

plt.show()

In [ ]:
# Plot Accuracy
plt.plot(history2.history['accuracy'], label='Train Accuracy')
plt.plot(history2.history['val_accuracy'], label='Validation Accuracy')
plt.title('Scratch Model Oversampling Accuracy')
plt.legend()

plt.show()

### **Evaluasi Scratch Model Oversampling**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

y_true_sc2 = []
y_pred_sc2 = []

for images, labels in test_ds:
    preds_sc2 = model_base2.predict(images, verbose=0)
    pred_labels_sc2 = np.argmax(preds_sc2, axis=1)

    y_true_sc2.extend(labels.numpy())
    y_pred_sc2.extend(pred_labels_sc2)

acc_sc2 = accuracy_score(y_true_sc2, y_pred_sc2)
precision_sc2 = precision_score(y_true_sc2, y_pred_sc2, average='macro', zero_division=0)
recall_sc2 = recall_score(y_true_sc2, y_pred_sc2, average='macro', zero_division=0)
f1_sc2 = f1_score(y_true_sc2, y_pred_sc2, average='macro', zero_division=0)

print("=== Scratch Model Oversampling ===")
print("Accuracy :", acc_sc2)
print("Precision:", precision_sc2)
print("Recall   :", recall_sc2)
print("F1-Score :", f1_sc2)

print(classification_report(y_true_sc2, y_pred_sc2, target_names=classes, zero_division=0))

## **Transfer Model (No Oversampling, Class Weight, EfficientNetB1)**

---



In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input
def prepare_data(dataset, augment=False):
    dataset = dataset.map(
        lambda x, y: (preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if augment:
        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_ds_mod1 = prepare_data(raw_train_ds, augment=True)
val_ds_mod1   = prepare_data(raw_val_ds, augment=False)
test_ds_mod1  = prepare_data(raw_test_ds, augment=False)

In [ ]:
base_eff = tf.keras.applications.EfficientNetB1(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_eff.trainable = False

In [ ]:
model_Modified1 = models.Sequential([
    base_eff,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(4, activation='softmax')
])
model_Modified1.summary()

In [ ]:
model_Modified1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history3 = model_Modified1.fit(
    train_ds_mod1,
    validation_data=val_ds_mod1,
    epochs=50,
    class_weight=class_weight,
    callbacks=[lr_scheduler, early_stop],
    verbose=1
)

In [ ]:
# Plot Train
plt.plot(history3.history['loss'], label='Train Loss')
plt.plot(history3.history['val_loss'], label='Validation Loss')
plt.title('Modified Model No Oversampling(Class Weight) Loss')
plt.legend()

plt.show()

In [ ]:
# Plot Accuracy
plt.plot(history3.history['accuracy'], label='Train Accuracy')
plt.plot(history3.history['val_accuracy'], label='Validation Accuracy')
plt.title('Modified Model No Oversampling(Class Weight) Accuracy')
plt.legend()

plt.show()

### **Evaluasi Transfer Model No Oversampling**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

y_true_mod1 = []
y_pred_mod1 = []

for images, labels in test_ds_mod1:
    preds_mod1 = model_Modified1.predict(images, verbose=0)
    pred_labels_mod1 = np.argmax(preds_mod1, axis=1)

    y_true_mod1.extend(labels.numpy())
    y_pred_mod1.extend(pred_labels_mod1)

acc_mod1 = accuracy_score(y_true_mod1, y_pred_mod1)
precision_mod1 = precision_score(y_true_mod1, y_pred_mod1, average='macro', zero_division=0)
recall_mod1 = recall_score(y_true_mod1, y_pred_mod1, average='macro', zero_division=0)
f1_mod1 = f1_score(y_true_mod1, y_pred_mod1, average='macro', zero_division=0)

print("=== Modified Model No Oversampling (Class Weight) ===")
print("Accuracy :", acc_mod1)
print("Precision:", precision_mod1)
print("Recall   :", recall_mod1)
print("F1-Score :", f1_mod1)

print(classification_report(y_true_mod1, y_pred_mod1, target_names=classes, zero_division=0))

## **Transfer Model (Oversampling, EfficientNetB1)**

---



In [ ]:
def split_classes(dataset):
    dataset = dataset.unbatch()

    c0 = dataset.filter(lambda x, y: y == 0)
    c1 = dataset.filter(lambda x, y: y == 1)
    c2 = dataset.filter(lambda x, y: y == 2)
    c3 = dataset.filter(lambda x, y: y == 3)

    return c0, c1, c2, c3

In [ ]:
def make_oversampled_ds(dataset):
    c0, c1, c2, c3 = split_classes(dataset)

    c0 = c0.repeat()
    c1 = c1.repeat()
    c2 = c2.repeat()
    c3 = c3.repeat()

    balanced = tf.data.Dataset.sample_from_datasets(
        [c0, c1, c2, c3],
        weights=[0.25, 0.25, 0.25, 0.25],
        seed=SEED
    )

    return balanced

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

def prepare_data(dataset, augment=False, oversample=False):

    if oversample:
        # make_oversampled_ds menghasilkan data unbatch
        dataset = make_oversampled_ds(dataset)
        dataset = dataset.shuffle(1000, seed=SEED)
        dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.map(
        lambda x, y: (preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if augment:
        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds_mod2 = prepare_data(raw_train_ds, augment=True, oversample=True)
val_ds_mod2   = prepare_data(raw_val_ds, augment=False, oversample=False)
test_ds_mod2  = prepare_data(raw_test_ds, augment=False, oversample=False)

In [ ]:
for images, labels in train_ds_mod2.take(1):
    print("Train shape:", images.shape, labels.shape)

for images, labels in val_ds_mod2.take(1):
    print("Val shape:", images.shape, labels.shape)

In [ ]:
base_eff2 = tf.keras.applications.EfficientNetB1(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_eff2.trainable = False

In [ ]:
model_Modified2 = models.Sequential([
    base_eff2,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(4, activation='softmax')
])
model_Modified2.summary()

In [ ]:
model_Modified2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history4 = model_Modified2.fit(
    train_ds_mod2,
    validation_data=val_ds_mod2,
    epochs=50,
    steps_per_epoch=len(raw_train_ds),
    callbacks=[lr_scheduler, early_stop],
    verbose=1
)

In [ ]:
# Plot Train
plt.plot(history4.history['loss'], label='Train Loss')
plt.plot(history4.history['val_loss'], label='Validation Loss')
plt.title('Modified Model Oversampling Loss')
plt.legend()

plt.show()

In [ ]:
# Plot Accuracy
plt.plot(history4.history['accuracy'], label='Train Accuracy')
plt.plot(history4.history['val_accuracy'], label='Validation Accuracy')
plt.title('Modified Model Oversampling Accuracy')
plt.legend()

plt.show()

### **Evaluasi Transfer Model Oversampling**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

y_true_mod2 = []
y_pred_mod2 = []

for images, labels in test_ds_mod2:
    preds_mod2 = model_Modified2.predict(images, verbose=0)
    pred_labels_mod2 = np.argmax(preds_mod2, axis=1)

    y_true_mod2.extend(labels.numpy())
    y_pred_mod2.extend(pred_labels_mod2)

acc_mod2 = accuracy_score(y_true_mod2, y_pred_mod2)
precision_mod2 = precision_score(y_true_mod2, y_pred_mod2, average='macro', zero_division=0)
recall_mod2 = recall_score(y_true_mod2, y_pred_mod2, average='macro', zero_division=0)
f1_mod2 = f1_score(y_true_mod2, y_pred_mod2, average='macro', zero_division=0)

print("=== Modified Model Oversampling ===")
print("Accuracy :", acc_mod2)
print("Precision:", precision_mod2)
print("Recall   :", recall_mod2)
print("F1-Score :", f1_mod2)

print(classification_report(y_true_mod2, y_pred_mod2, target_names=classes, zero_division=0))

## **Comparasion**

In [ ]:
print("Scratch Model No Oversampling (Class Weight)")
print(classification_report(y_true_sc1, y_pred_sc1, target_names=classes, zero_division=0))
print()
print("Scratch Model Oversampling")
print(classification_report(y_true_sc2, y_pred_sc2, target_names=classes, zero_division=0))
print()
print("Transfer Model No Oversampling (Class Weight, EfficientNetB1)")
print(classification_report(y_true_mod1, y_pred_mod1, target_names=classes, zero_division=0))
print()
print("Transfer Model Oversampling (EfficientNetB1)")
print(classification_report(y_true_mod2, y_pred_mod2, target_names=classes, zero_division=0))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Membuat fungsi untuk visualisasi Confusion Matrix
def plot_confusion_matrix(y_true, y_pred, title):
    # Menghitung confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Mengatur ukuran figur
    plt.figure(figsize=(8, 6))

    # Membuat heatmap menggunakan seaborn
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)

    # Menambahkan label dan judul
    plt.title(f'Confusion Matrix: {title}', fontsize=14, pad=15)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)

    # Menampilkan plot
    plt.tight_layout()
    plt.show()

# 1. Scratch Model No Oversampling (Class Weight)
plot_confusion_matrix(y_true_sc1, y_pred_sc1, "Scratch Model No Oversampling")

# 2. Scratch Model Oversampling
plot_confusion_matrix(y_true_sc2, y_pred_sc2, "Scratch Model Oversampling")

# 3. Transfer Model No Oversampling (EfficientNetB1)
plot_confusion_matrix(y_true_mod1, y_pred_mod1, "Transfer Model No Oversampling")

# 4. Transfer Model Oversampling (EfficientNetB1)
plot_confusion_matrix(y_true_mod2, y_pred_mod2, "Transfer Model Oversampling")